# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector Construction

The feature vector is prepared using historical February 2026 content performance data.

The following numerical features are included:

- `feb_impressions` — Total impressions during February.
- `feb_clicks` — Total clicks during February.
- `feb_avg_position` — Average search position during February.
- `feb_ctr` — Click-through rate calculated from clicks and impressions.
- `feb_log_impressions` — Log-transformed impressions.
- `feb_log_clicks` — Log-transformed clicks.

Missing numerical values are handled using median imputation. If categorical columns are available, missing values are filled with `Unknown` and categorical values are converted using one-hot encoding.

Client and content IDs are retained as metadata for grouping and analysis, not used as predictive features.

The March target is kept separate from the feature vector to avoid target leakage. Only information available before the prediction period should be used as model input.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build the feature vector using information available in February


import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Connect to Hugging Face warehouse
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Define warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

# Aggregate February performance
feb_monthly = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS feb_impressions,
    SUM(gsc_clicks) AS feb_clicks,
    AVG(gsc_avg_position) AS feb_avg_position
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

# Aggregate March performance for target creation only
mar_monthly = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS mar_clicks
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

# Match February and March records
data = feb_monthly.merge(
    mar_monthly,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Create target separately
data["target"] = (
    data["mar_clicks"] < data["feb_clicks"]
).astype(int)

print("Matched records:", len(data))
print("February features loaded successfully.")
print("Target created separately.")

# Build engineered features using February data only
X = data[
    [
        "feb_impressions",
        "feb_clicks",
        "feb_avg_position"
    ]
].copy()

X["feb_ctr"] = (
    X["feb_clicks"] /
    X["feb_impressions"].replace(0, np.nan)
)

X["feb_log_impressions"] = np.log1p(
    X["feb_impressions"].clip(lower=0)
)

X["feb_log_clicks"] = np.log1p(
    X["feb_clicks"].clip(lower=0)
)

# Handle missing and infinite values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Keep identifiers as metadata, not predictive features
metadata = data[
    ["client_hash_id", "content_hash_id"]
].copy()

y = data["target"].copy()

print("\nFeature vector shape:", X.shape)
print("Missing feature values:", X.isna().sum().sum())
print("Target shape:", y.shape)

display(X.head())


Matched records: 134238
February features loaded successfully.
Target created separately.

Feature vector shape: (134238, 6)
Missing feature values: 0
Target shape: (134238,)


,feb_impressions,feb_clicks,feb_avg_position,feb_ctr,feb_log_impressions,feb_log_clicks
0,5.0,1.0,5.000000,0.200000,1.791759,0.693147
1,96.0,0.0,5.262333,0.000000,4.574711,0.000000
2,235.0,0.0,6.407819,0.000000,5.463832,0.000000
3,18.0,0.0,6.961538,0.000000,2.944439,0.000000
4,74.0,1.0,8.063910,0.013514,4.317488,0.693147


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing-value handling | Type | Available before prediction? |
|---|---|---|---|---|
| `feb_impressions` | Total search impressions in February 2026 | Median imputation | Numerical | Yes |
| `feb_clicks` | Total search clicks in February 2026 | Median imputation | Numerical | Yes |
| `feb_avg_position` | Average search position in February 2026 | Median imputation | Numerical | Yes |
| `feb_ctr` | February clicks divided by February impressions | Invalid or missing values handled before imputation | Numerical | Yes |
| `feb_log_impressions` | Log-transformed February impressions | Median imputation | Numerical | Yes |
| `feb_log_clicks` | Log-transformed February clicks | Median imputation | Numerical | Yes |

**Categorical features:** No categorical predictive features are included. Client and content IDs are retained for record matching but excluded from model inputs.

**Availability:** All six features are calculated using February 2026 data, so they are available before the March 2026 prediction period.

**Target separation:** March clicks are used only to create the target variable. They are not included in the feature vector.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage audit

The target is created using March 2026 clicks compared with February 2026 clicks.

The predictive features are derived only from February 2026 data. March clicks are kept separate and are not included in the feature vector.

The audit checks for:
- Target-derived columns accidentally included in the features.
- Future-period information from March 2026.
- Identifier columns that could act as shortcuts.
- Features that may have a mathematical relationship with the target.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Check whether target or future information entered X

print("FEATURES USED:")
print(list(X.columns))

print("\nTARGET INCLUDED IN FEATURES?")
print("target" in X.columns)

# 2. Check for March/future columns in X

future_features = [
    col for col in X.columns
    if "mar" in col.lower() or "march" in col.lower()
]

print("\nFUTURE-DATA FEATURES:", future_features)

# 3. Check whether identifiers entered X

identifier_features = [
    col for col in X.columns
    if col in ["client_hash_id", "content_hash_id"]
]

print("\nIDENTIFIER FEATURES:", identifier_features)

# 4. Check the target definition

print("\nTARGET DEFINITION:")
print("target = (mar_clicks < feb_clicks)")

# 5. Final checks

assert "target" not in X.columns
assert len(future_features) == 0
assert len(identifier_features) == 0

print("\nLeakage checks passed.")

FEATURES USED:
['feb_impressions', 'feb_clicks', 'feb_avg_position', 'feb_ctr', 'feb_log_impressions', 'feb_log_clicks']

TARGET INCLUDED IN FEATURES?
False

FUTURE-DATA FEATURES: []

IDENTIFIER FEATURES: []

TARGET DEFINITION:
target = (mar_clicks < feb_clicks)

Leakage checks passed.


In [8]:
# 6. Verify target is created only from the intended label columns

expected_target = (data["mar_clicks"] < data["feb_clicks"]).astype(int)

print("TARGET DEFINITION MATCHES:", (y == expected_target).all())

# 7. Check engineered features

expected_ctr = data["feb_clicks"] / data["feb_impressions"]

valid_rows = data["feb_impressions"] > 0

ctr_check = np.allclose(
    X.loc[valid_rows, "feb_ctr"],
    expected_ctr.loc[valid_rows],
    equal_nan=True
)

print("CTR DERIVED FROM FEBRUARY DATA:", ctr_check)

# 8. Check whether any unexpected columns entered the feature vector

allowed_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_ctr",
    "feb_log_impressions",
    "feb_log_clicks"
]

print("ONLY APPROVED FEATURES USED:", set(X.columns) == set(allowed_features))

# 9. Document the February clicks relationship

print("\nIMPORTANT OBSERVATION:")
print("feb_clicks is both a predictive feature and part of the target definition.")
print("This is not future leakage, but it creates a mathematical relationship with the target.")

assert (y == expected_target).all()
assert ctr_check
assert set(X.columns) == set(allowed_features)

print("\nSECOND AUDIT PASSED.")

TARGET DEFINITION MATCHES: True
CTR DERIVED FROM FEBRUARY DATA: True
ONLY APPROVED FEATURES USED: True

IMPORTANT OBSERVATION:
feb_clicks is both a predictive feature and part of the target definition.
This is not future leakage, but it creates a mathematical relationship with the target.

SECOND AUDIT PASSED.


### Leakage audit conclusion

The feature vector passed the automated leakage checks.

No target column, March-period feature, or identifier column was found in the predictive feature set. All six features are derived from February 2026 data and are available before the March prediction period.

The target definition was independently verified, and the engineered CTR feature was confirmed to use February data.

One important consideration is that `feb_clicks` is used both as a predictive feature and in the target definition. Although this is not temporal leakage, it creates a mathematical relationship between the feature and target and may influence model performance.

Therefore, the feature vector is free from the tested forms of future-data and direct target leakage, but the relationship between February clicks and the target must be considered during model evaluation.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields and reasons

| Excluded field | Reason for exclusion |
|---|---|
| `mar_clicks` | Used to create the target; including it would cause direct target leakage. |
| `mar_impressions` | March-period information is unavailable at the February prediction point. |
| `client_hash_id` | Identifier used for grouping and record matching, not a meaningful predictive feature. |
| `content_hash_id` | Content identifier that could encourage memorization rather than learning general patterns. |
| `target` | Label column used as the prediction outcome, not an input feature. |

### Privacy considerations

Hashed client and content identifiers are retained only for matching, grouping, and analysis. They are excluded from the predictive feature vector to reduce identifier-based memorization and unnecessary use of identifiers.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.